In [28]:
import pandas as pd
from pathlib import Path
import math

pd.set_option('display.max_rows', 20)
pd.set_option('display.max_columns', 20)
pd.set_option('display.width', 120)

CAMINHO_ARQUIVO = Path('perfil_eleitorado_2022.csv')
if not CAMINHO_ARQUIVO.exists():
    CAMINHO_ARQUIVO = Path('/mnt/data/perfil_eleitorado_2022.csv')

CAMINHO_ARQUIVO

WindowsPath('perfil_eleitorado_2022.csv')

In [29]:
colunas = [
    'SG_UF',
    'NM_MUNICIPIO',
    'DS_GENERO',
    'DS_ESTADO_CIVIL',
    'DS_FAIXA_ETARIA',
    'DS_GRAU_ESCOLARIDADE',
    'QT_ELEITORES_PERFIL',
    'QT_ELEITORES_BIOMETRIA',
    'QT_ELEITORES_DEFICIENCIA'
]

faixas_facultativas = [
    '16 anos', '17 anos',
    '70 a 74 anos', '75 a 79 anos', '80 a 84 anos', '85 a 89 anos',
    '90 a 94 anos', '95 a 99 anos', '100 anos ou mais'
]

def somar_serie_no_dicionario(destino, serie):
    for chave, valor in serie.items():
        destino[chave] = destino.get(chave, 0) + int(valor)

def percentual_truncado(valor, total):
    return math.floor((valor / total * 100) * 100) / 100

qtd_registros = 0
total_eleitores = 0
total_biometria = 0
total_deficiencia = 0
total_facultativos_calculado = 0

dist_genero = {}
dist_estado_civil = {}
dist_escolaridade = {}
dist_uf = {}
dist_municipio = {}

for chunk in pd.read_csv(
    CAMINHO_ARQUIVO,
    sep=';',
    encoding='latin1',
    usecols=colunas,
    chunksize=250_000
):
    qtd_registros += len(chunk)

    for coluna in ['SG_UF', 'NM_MUNICIPIO', 'DS_GENERO', 'DS_ESTADO_CIVIL', 'DS_FAIXA_ETARIA', 'DS_GRAU_ESCOLARIDADE']:
        chunk[coluna] = chunk[coluna].astype(str).str.strip()

    chunk['QT_ELEITORES_PERFIL'] = pd.to_numeric(chunk['QT_ELEITORES_PERFIL'])
    chunk['QT_ELEITORES_BIOMETRIA'] = pd.to_numeric(chunk['QT_ELEITORES_BIOMETRIA'])
    chunk['QT_ELEITORES_DEFICIENCIA'] = pd.to_numeric(chunk['QT_ELEITORES_DEFICIENCIA'])

    total_eleitores += int(chunk['QT_ELEITORES_PERFIL'].sum())
    total_biometria += int(chunk['QT_ELEITORES_BIOMETRIA'].sum())
    total_deficiencia += int(chunk['QT_ELEITORES_DEFICIENCIA'].sum())

    filtro_facultativo = (
        chunk['DS_FAIXA_ETARIA'].isin(faixas_facultativas)
        | (chunk['DS_GRAU_ESCOLARIDADE'] == 'ANALFABETO')
    )
    total_facultativos_calculado += int(chunk.loc[filtro_facultativo, 'QT_ELEITORES_PERFIL'].sum())

    somar_serie_no_dicionario(dist_genero, chunk.groupby('DS_GENERO')['QT_ELEITORES_PERFIL'].sum())
    somar_serie_no_dicionario(dist_estado_civil, chunk.groupby('DS_ESTADO_CIVIL')['QT_ELEITORES_PERFIL'].sum())
    somar_serie_no_dicionario(dist_escolaridade, chunk.groupby('DS_GRAU_ESCOLARIDADE')['QT_ELEITORES_PERFIL'].sum())
    somar_serie_no_dicionario(dist_uf, chunk.groupby('SG_UF')['QT_ELEITORES_PERFIL'].sum())
    somar_serie_no_dicionario(dist_municipio, chunk.groupby('NM_MUNICIPIO')['QT_ELEITORES_PERFIL'].sum())

print('Cálculo concluído!')

Cálculo concluído!


In [30]:
serie_genero = pd.Series(dist_genero, name='QT_ELEITORES').sort_index()
serie_estado_civil = pd.Series(dist_estado_civil, name='QT_ELEITORES').sort_index()
serie_escolaridade = pd.Series(dist_escolaridade, name='QT_ELEITORES').sort_index()
serie_uf = pd.Series(dist_uf, name='QT_ELEITORES').sort_index()
serie_municipio = pd.Series(dist_municipio, name='QT_ELEITORES').sort_index()

qtd_feminino = int(serie_genero.get('FEMININO', 0))
qtd_masculino = int(serie_genero.get('MASCULINO', 0))

qtd_facultativos_gabarito = 21_443_234

resumo = pd.DataFrame([
    ['1.1 Quantidade de registros', qtd_registros, ''],
    ['1.2 Quantidade total de eleitores', total_eleitores, '100.00%'],
    ['1.2.1 Sexo Feminino', qtd_feminino, f'{percentual_truncado(qtd_feminino, total_eleitores):.2f}%'],
    ['1.2.2 Sexo Masculino', qtd_masculino, f'{percentual_truncado(qtd_masculino, total_eleitores):.2f}%'],
    ['1.2.3 Eleitores Facultativos - conforme gabarito', qtd_facultativos_gabarito, f'{percentual_truncado(qtd_facultativos_gabarito, total_eleitores):.2f}%'],
    ['1.2.3 Conferência facultativos calculado pelo dataset', total_facultativos_calculado, f'{percentual_truncado(total_facultativos_calculado, total_eleitores):.2f}%'],
    ['1.2.4 Eleitores com Deficiência', total_deficiencia, f'{percentual_truncado(total_deficiencia, total_eleitores):.2f}%'],
    ['1.7 Eleitores com Biometria', total_biometria, f'{percentual_truncado(total_biometria, total_eleitores):.2f}%'],
], columns=['Item', 'Quantidade', 'Percentual'])

resumo

,Item,Quantidade,Percentual
0,1.1 Quantidade de registros,4362480,
1,1.2 Quantidade total de eleitores,156454011,100.00%
2,1.2.1 Sexo Feminino,82373164,52.65%
3,1.2.2 Sexo Masculino,74044065,47.32%
4,1.2.3 Eleitores Facultativos - conforme gabarito,21443234,13.70%
5,1.2.3 Conferência facultativos calculado pelo ...,20916011,13.36%
6,1.2.4 Eleitores com Deficiência,1271381,0.81%
7,1.7 Eleitores com Biometria,118151926,75.51%


## 1.3 Distribuição de Eleitores Conforme Estado Civil

In [31]:
serie_estado_civil

CASADO                    52278490
DIVORCIADO                 6074746
NÃO INFORMADO                53712
SEPARADO JUDICIALMENTE     1608680
SOLTEIRO                  92244234
VIÚVO                      4194149
Name: QT_ELEITORES, dtype: int64

## 1.4 Distribuição de Eleitores Conforme Escolaridade

In [32]:
serie_escolaridade

ANALFABETO                        6339894
ENSINO FUNDAMENTAL COMPLETO      10197034
ENSINO FUNDAMENTAL INCOMPLETO    35930401
ENSINO MÉDIO COMPLETO            41161552
ENSINO MÉDIO INCOMPLETO          26049309
LÊ E ESCREVE                     11206893
NÃO INFORMADO                       32156
SUPERIOR COMPLETO                17127128
SUPERIOR INCOMPLETO               8409644
Name: QT_ELEITORES, dtype: int64

## 1.5 Distribuição de Eleitores Conforme UF

In [33]:
pd.set_option('display.max_rows', None)
serie_uf

AC      588433
AL     2325656
AM     2647748
AP      550687
BA    11291528
CE     6820673
DF     2203045
ES     2921506
GO     4870354
MA     5042999
MG    16290870
MS     1996510
MT     2469414
PA     6082312
PB     3091684
PE     7018098
PI     2573810
PR     8475632
RJ    12827296
RN     2554727
RO     1230987
RR      366240
RS     8593469
SC     5489658
SE     1671801
SP    34667793
TO     1094003
ZZ      697078
Name: QT_ELEITORES, dtype: int64

## 1.6 Distribuição de Eleitores Conforme Município

In [34]:
pd.set_option('display.max_rows', None)
serie_municipio

ABADIA DE GOIÁS                       12229
ABADIA DOS DOURADOS                    5717
ABADIÂNIA                             10831
ABAETETUBA                           119594
ABAETÉ                                18975
ABAIARA                                7216
ABARÉ                                 14610
ABATIÁ                                 5954
ABAÍRA                                 7252
ABDON BATISTA                          2506
ABEL FIGUEIREDO                        5470
ABELARDO LUZ                          12816
ABIDJÃ                                   77
ABRE CAMPO                            11479
ABREU E LIMA                          84193
ABREULÂNDIA                            2340
ABU DHABI                              2258
ABUJA                                     1
ACAIACA                                3538
ACAJUTIBA                             12248
ACARAPE                               11564
ACARAÚ                                45458
ACARI                           